# HW05 — Анализ ошибок / Error analysis
**100 баллов: код 80 + исследование 20. / 100 points: code 80 + analysis 20.** Colab CPU, 120–180 минут / minutes.

Реальные данные: [Wine, sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html), [UCI Wine](https://archive.ics.uci.edu/dataset/109/wine). Набор содержит 178 наблюдений, 13 химических измерений и 3 класса сортов винограда. Это классификация сорта, не прогноз оценки качества вина. Одно наблюдение — образец вина; признаки доступны после лабораторного анализа. / 178 wine samples, 13 chemical measurements, three cultivar classes. Predict cultivar, not wine quality. One observation is a wine sample; features become available after laboratory analysis.

Стратифицированное разбиение по образцам 60/20/20; нет ID производителя, партии или времени. Не делать выводы о новых производителях/годах. Не придумывать группы. / Stratified sample split, approximately 60/20/20; no producer, batch or time IDs. Do not claim generalization to new producers or vintages, and do not invent groups.

Заполните имя и TODO. Перезапустите и выполните все ячейки. Не меняйте split, тесты, модели и правило выбора после доступа к Test. / Enter your name and complete TODOs. Restart and run all cells. Do not change splits, checks, models or selection rule after inspecting Test.

In [ ]:
STUDENT_NAME = ''
STUDENT_GROUP = ''

In [ ]:
import json, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
SEED = 42

In [ ]:
data=load_wine();X,y=data.data,data.target;labels=np.arange(3)
ids = np.arange(len(y))
trainval, test = train_test_split(ids, test_size=.2, random_state=SEED, stratify=y)
train, val = train_test_split(trainval, test_size=.25, random_state=SEED, stratify=y[trainval])
assert not (set(train)&set(val) or set(train)&set(test) or set(val)&set(test))
assert sorted(np.r_[train,val,test]) == ids.tolist()
print({k: len(v) for k,v in {'train':train,'validation':val,'test':test}.items()})
# Prespecified baseline and candidate; only max_depth changes.
baseline=DecisionTreeClassifier(max_depth=2,min_samples_leaf=3,random_state=SEED).fit(X[train],y[train])
pred_val=baseline.predict(X[val])
cm=confusion_matrix(y[val],pred_val,labels=labels)
ConfusionMatrixDisplay(cm,display_labels=data.target_names).plot();plt.show()

## T1 — 20 баллов / points
`class_report(matrix)`: квадратная матрица K×K целых неотрицательных counts; строки — true, столбцы — predicted. Верните dict с массивами support, precision, recall, f1 (K элементов). Нулевые знаменатели → 0. NumPy, без sklearn metrics. Не менять вход. Валидация входного контракта не требуется. / Given a square K×K nonnegative integer confusion matrix (true rows, predicted columns), return arrays support, precision, recall, f1. Zero denominators give 0. NumPy only, no sklearn metrics, no input mutation; input validation is not required.

In [ ]:
def class_report(matrix):
    raise NotImplementedError("TODO")

## T2 — 15 баллов / points
`error_ids(row_ids,y_true,y_pred)`: одинаковые непустые 1D массивы, уникальные целые row_ids. Верните только ID ошибок, в исходном порядке, не позиции и не метки. Не менять вход. / Equal-length nonempty 1D arrays with unique integer IDs. Return misclassified row IDs in original order, not positions or labels; no input mutation.

In [ ]:
def error_ids(row_ids, y_true, y_pred):
    raise NotImplementedError("TODO")

## T3 — 15 баллов / points
`slice_report(y_true,y_pred,mask)`: одинаковые 1D массивы, mask булев. Верните dict: n (Python int), errors (int), error_rate (float). Для пустого среза error_rate=None. Не менять вход. / Equal-length 1D arrays; boolean mask. Return n, errors as Python integers and error_rate as float, or None for an empty slice. No input mutation.

In [ ]:
def slice_report(y_true,y_pred,mask):
    raise NotImplementedError("TODO")

## T4 — 15 баллов / points
`select_model(scores)`: dict с ключами baseline и candidate, конечными macro-F1 в [0,1]. Максимальный score; точное равенство → baseline. Верните имя; не менять dict. / Dict containing baseline and candidate finite macro-F1 scores in [0,1]. Select maximum; exact ties retain baseline. Return the name without mutation.

In [ ]:
def select_model(scores):
    raise NotImplementedError("TODO")

## T5 — 15 баллов / points
`make_log(train_ids,val_ids,test_ids,selected,validation_scores)`: JSON dict с assignment="HW05_Error_Analysis", dataset="sklearn.load_wine/UCI Wine", seed=42, train_ids, val_ids, test_ids (Python int lists, порядок сохраняется), selected, validation_scores, numpy_version и sklearn_version (реальные версии). Никаких дат из локальных часов. / Return a JSON-serializable dict with the listed exact fields, ID order preserved and actual installed versions. Do not invent a submission timestamp.

In [ ]:
def make_log(train_ids,val_ids,test_ids,selected,validation_scores):
    raise NotImplementedError("TODO")

## Исследование / Analysis — 20
1. (5) Таблица counts и recall всех трёх классов (2), слабый класс с denominator (1), два ID ошибок с признаками и различием наблюдения/гипотезы (2). Если ошибок меньше двух, укажите фактическое количество. / Counts and recall for all classes (2), weakest class with denominator (1), two error IDs with features and observation versus hypothesis (2); if fewer errors exist, report the actual count.
2. (5) Срез по признаку `alcohol`: порог — медиана Train (2). Для low/high укажите n, errors, rate (2), ограничение малого среза (1). / Slice by alcohol using training median (2); n/errors/rate for low/high (2), small-slice limitation (1).
3. (6) **До запуска candidate** запишите проверяемую гипотезу для depth 2→4 (2). Выполните сравнение validation macro-F1, примените T4 (2). После фиксации выбора один раз оцените Test и объясните результат, даже при ухудшении (2). / Before running candidate, write a testable hypothesis for depth 2→4 (2). Compare validation macro-F1 and apply T4 (2). After freezing selection, evaluate Test once and interpret even a worse result (2).
4. (4) Журнал с версиями и ID (1), источник времени — событие загрузки Drive, не дата из ноутбука (1), источники/AI-помощь (1), ограничение переносимости Wine (1). / Log with versions and IDs (1), Drive upload event as submission-time source (1), sources/AI assistance (1), Wine generalization limitation (1).

In [ ]:
# TODO: use class_report(cm), error_ids(val,y[val],pred_val), slice_report(...).
# Fit the alcohol median on X[train,0], not all data.
# Write your hypothesis BEFORE training the candidate.
hypothesis = ''
analysis = '' 

## Validation → заморозка → Test / Validation → freeze → Test
Ячейка выполняется после T1–T5 и записи гипотезы. Дополните анализ выше. Test только после выбора. / Run after completing T1–T5 and writing the hypothesis. Complete the analysis above; Test only after selection.

In [ ]:
try:
    display(pd.DataFrame(class_report(cm),index=data.target_names))
    print('Error IDs:',error_ids(val,y[val],pred_val))
    cut=float(np.median(X[train,0]))
    print('Alcohol cutoff:',cut)
    for name,mask in {'low':X[val,0]<=cut,'high':X[val,0]>cut}.items():
        print(name,slice_report(y[val],pred_val,mask))
    candidate=DecisionTreeClassifier(max_depth=4,min_samples_leaf=3,random_state=SEED).fit(X[train],y[train])
    scores={name:float(f1_score(y[val],model.predict(X[val]),average='macro')) for name,model in {'baseline':baseline,'candidate':candidate}.items()}
    selected=select_model(scores)
    log=make_log(train,val,test,selected,scores)
    print('Validation:',scores,'Selected:',selected)
    # Selection frozen BEFORE test. Do not optimize further from these results.
    final_model={'baseline':baseline,'candidate':candidate}[selected]
    log['test_macro_f1']=float(f1_score(y[test],final_model.predict(X[test]),average='macro'))
    log['baseline_test_macro_f1']=float(f1_score(y[test],baseline.predict(X[test]),average='macro'))
    log['data_sha256']=hashlib.sha256(X.tobytes()+y.tobytes()).hexdigest()
    print(json.dumps(log,ensure_ascii=False))
except NotImplementedError:
    print('Complete T1–T5 / Завершите T1–T5')

## Самопроверка кода / Code self-check
Проверки используют реальные validation-предсказания Wine и их срезы. 80 баллов — только код; объяснения проверяет преподаватель. Не изменяйте эту ячейку. / Checks use real Wine validation predictions and subsets. 80 points cover code only; explanations are assessed separately. Do not edit this cell.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
score=0; tasks=[]
def check_t1():
    for yy,pp in [(y[val],pred_val),(y[val][y[val]!=2],pred_val[y[val]!=2])]:
        m=confusion_matrix(yy,pp,labels=labels);before=m.copy();r=class_report(m)
        p,re,f,s=precision_recall_fscore_support(yy,pp,labels=labels,zero_division=0)
        for key,expected in [('support',s),('precision',p),('recall',re),('f1',f)]:
            assert np.asarray(r[key]).shape==(3,) and np.allclose(r[key],expected)
        assert np.array_equal(m,before)
def check_t2():
    for ix in [np.arange(len(val)),np.arange(len(val))[::-1]]:
        assert np.array_equal(error_ids(val[ix],y[val][ix],pred_val[ix]),val[ix][y[val][ix]!=pred_val[ix]])
def check_t3():
    for mask in [X[val,0]<=np.median(X[train,0]), X[val,0]>np.median(X[train,0]), np.zeros(len(val),dtype=bool)]:
        r=slice_report(y[val],pred_val,mask);n=int(mask.sum());e=int((y[val][mask]!=pred_val[mask]).sum())
        assert type(r['n']) is int and type(r['errors']) is int and (r['n'],r['errors'])==(n,e)
        assert r['error_rate'] is None if n==0 else isinstance(r['error_rate'],float) and np.isclose(r['error_rate'],e/n)
def check_t4():
    # Scores measured on actual validation predictions; swap role names to test both directions.
    a=float(f1_score(y[val],pred_val,average='macro'))
    b=float(f1_score(y[val],baseline.predict(X[val]),average='micro'))
    for s in [{'baseline':a,'candidate':b},{'baseline':b,'candidate':a},{'baseline':a,'candidate':a}]:
        before=s.copy();assert select_model(s)==('candidate' if s['candidate']>s['baseline'] else 'baseline');assert s==before
def check_t5():
    s={'baseline':float(f1_score(y[val],pred_val,average='macro')),'candidate':float(f1_score(y[val],pred_val,average='macro'))}
    r=make_log(train,val,test,'baseline',s);json.dumps(r)
    assert r['assignment']=='HW05_Error_Analysis' and r['dataset']=='sklearn.load_wine/UCI Wine' and r['seed']==42
    for k,v in [('train_ids',train),('val_ids',val),('test_ids',test)]:
        assert r[k]==v.tolist() and all(type(i) is int for i in r[k])
    assert r['selected']=='baseline' and r['validation_scores']==s
    assert r['numpy_version']==np.__version__ and r['sklearn_version']==sklearn.__version__
for name,points,fn in [('T1',20,check_t1),('T2',15,check_t2),('T3',15,check_t3),('T4',15,check_t4),('T5',15,check_t5)]:
    try:fn();got=points;status='OK'
    except Exception as e:got=0;status=type(e).__name__
    score+=got;tasks.append({'task':name,'score':got,'max':points,'status':status});print(name,status,got)
print(json.dumps({'name':STUDENT_NAME,'group':STUDENT_GROUP,'assignment':'HW05_Error_Analysis','code_score':score,'score_kind':'code_only','explanation_score':None,'total_score':None,'tasks':tasks},ensure_ascii=False))

Сдайте выполненный `.ipynb` в [общую папку](https://drive.google.com/drive/folders/1NNElGtsfynHtIHNngesVgpm71Hi9lkXm), имя `Группа_Фамилия_HW05.ipynb`. Срок — в матрице преподавателя; локальное время не используется. / Upload the executed notebook to the submission folder as `Group_Surname_HW05.ipynb`. Deadline comes from the instructor matrix; local time is not used.